# GRPO: inspect examples or the full training split

Set `NUM_EXAMPLES` to `1`–`5` for a walkthrough, or `None` for the full GSM8K training split. `BATCH_SIZE` controls how many prompts are processed at once. Each prompt produces `GROUP_SIZE` sampled completions.

This notebook inspects sampling, rewards, advantages, token log probabilities, ratios, and policy loss. It computes gradients for the current policy but does **not** update model weights.

In [33]:
import logging
import sys
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:

# Support a kernel started in this notebook's folder or at the repository root.
module_dir = Path.cwd()
if not (module_dir / "data.py").is_file():
    module_dir = module_dir / "model_training" / "post_training" / "06-grpo"
if not (module_dir / "data.py").is_file():
    raise FileNotFoundError("Start the kernel in 06-grpo or the deep_learning repository root.")
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from data import make_prompt
from model import inspect_grpo_batch

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## 1. Choose the examples and generation settings

For the complete training split, set `NUM_EXAMPLES = None` and `START_INDEX = 0`. A small batch limits memory use; a full run still requires generating and scoring every completion.

In [35]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
DATASET_NAME = "openai/gsm8k"
NUM_EXAMPLES = 1       # 1–5 for exploration; None for the full training split
START_INDEX = 0        # Dataset index of the first selected example
BATCH_SIZE = 1         # Prompts per batch, independent of GROUP_SIZE
GROUP_SIZE = 4         # Sampled completions per prompt
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.9
CLIP_EPS = 0.2

In [36]:

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
dtype = (
    torch.bfloat16 if device.type == "cuda"
    else torch.float16 if device.type == "mps"
    else torch.float32
)
{"device": str(device), "dtype": str(dtype), "batch_size": BATCH_SIZE, "group_size": GROUP_SIZE}

{'device': 'mps', 'dtype': 'torch.float16', 'batch_size': 1, 'group_size': 4}

In [37]:
if BATCH_SIZE < 1 or GROUP_SIZE < 1 or START_INDEX < 0:
    raise ValueError("BATCH_SIZE and GROUP_SIZE must be positive, and START_INDEX must be nonnegative")

In [38]:
if NUM_EXAMPLES is not None and NUM_EXAMPLES < 1:
    raise ValueError("NUM_EXAMPLES must be positive or None")

In [41]:
train_data = load_dataset(DATASET_NAME, "main", split="train")
if START_INDEX >= len(train_data):
    raise ValueError("START_INDEX is outside the training split")

In [42]:
end_index = len(train_data) if NUM_EXAMPLES is None else min(START_INDEX + NUM_EXAMPLES, len(train_data))
selected_data = train_data.select(range(START_INDEX, end_index))

In [43]:

pd.DataFrame([{
    "training_examples": len(train_data),
    "selected_examples": len(selected_data),
    "first_dataset_index": START_INDEX,
    "last_dataset_index": end_index - 1,
    "expected_completions": len(selected_data) * GROUP_SIZE,
}])

,training_examples,selected_examples,first_dataset_index,last_dataset_index,expected_completions
0,7473,1,0,0,4


## 2. Load the model and inspect the first prompt

In [44]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device).eval()

first_example = selected_data[0]
display(pd.DataFrame([first_example]))
print(make_prompt(first_example["question"]))

,question,answer
0,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...


Solve the following math problem.
Return only the final numerical answer.
Use the format: FINAL_ANSWER: <number>

Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?


## 3. Inspect the first batch

The helper runs each GRPO stage for this batch. Generated IDs, attention masks, rewards, advantages, log probabilities, and ratios are available in `first_batch`.

In [45]:
first_batch_size = min(BATCH_SIZE, len(selected_data))
first_examples = [selected_data[i] for i in range(first_batch_size)]

In [ ]:

first_batch = inspect_grpo_batch(
    model=model,
    tokenizer=tokenizer,
    examples=first_examples,
    group_size=GROUP_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    device=device,
    clip_eps=CLIP_EPS,
)

pd.DataFrame([{
    "input_ids_shape": tuple(first_batch["input_ids"].shape),
    "output_ids_shape": tuple(first_batch["output_ids"].shape),
    "logits_shape": first_batch["logits_shape"],
    "rewards_shape": tuple(first_batch["rewards"].shape),
    "advantages_shape": tuple(first_batch["advantages"].shape),
    "old_logps_require_grad": first_batch["old_requires_grad"],
    "current_logps_require_grad": first_batch["current_requires_grad"],
    "policy_loss": first_batch["loss"],
}])

## 4. Completions, rewards, and advantages

Rows are ordered by prompt and then by completion. Rewards and advantages have shape `(number of prompts, group size)`; advantages are normalized within each prompt's group.

In [ ]:
first_results = pd.DataFrame(first_batch["records"])
first_results["example_index"] += START_INDEX
first_results[["example_index", "completion_index", "question", "gold_answer", "prediction", "reward", "advantage", "batch_loss"]]

In [ ]:
display({"rewards": first_batch["rewards"], "advantages": first_batch["advantages"]})

## 5. Process the selected dataset in batches

The first batch is reused. Results for every completion are retained in `results_df`; token IDs, log probabilities, and ratios are stored per row so you can inspect any example. Logits and computation graphs are released after each batch.

In [ ]:
results = []
batch_diagnostics = []

for start in tqdm(range(0, len(selected_data), BATCH_SIZE), desc="GRPO inspection batches"):
    batch_examples = [selected_data[i] for i in range(start, min(start + BATCH_SIZE, len(selected_data)))]
    batch = first_batch if start == 0 else inspect_grpo_batch(
        model=model,
        tokenizer=tokenizer,
        examples=batch_examples,
        group_size=GROUP_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        device=device,
        clip_eps=CLIP_EPS,
    )
    for record in batch["records"]:
        results.append({**record, "example_index": START_INDEX + start + record["example_index"]})
    batch_diagnostics.append({
        "first_example_index": START_INDEX + start,
        "examples": len(batch_examples),
        "logits_shape": batch["logits_shape"],
        "policy_loss": batch["loss"],
        "old_logps_require_grad": batch["old_requires_grad"],
        "current_logps_require_grad": batch["current_requires_grad"],
    })

results_df = pd.DataFrame(results)
batch_diagnostics_df = pd.DataFrame(batch_diagnostics)
results_df[["example_index", "completion_index", "question", "gold_answer", "prediction", "reward", "advantage", "batch_loss"]]

In [ ]:
batch_diagnostics_df

## 6. Inspect tokens for any completion

Change the two indices to select a row from `results_df`. These arrays contain generated tokens through the first stop token, excluding prompt and padding positions.

In [ ]:
EXAMPLE_TO_INSPECT = START_INDEX
COMPLETION_TO_INSPECT = 0

matches = results_df.loc[
    (results_df["example_index"] == EXAMPLE_TO_INSPECT)
    & (results_df["completion_index"] == COMPLETION_TO_INSPECT)
]
if matches.empty:
    raise ValueError("The selected example and completion are not present in results_df")
row = matches.iloc[0]
display(row[["question", "gold_answer", "prediction", "reward", "advantage", "batch_loss"]])

pd.DataFrame({
    "token_id": row["token_ids"],
    "token": tokenizer.convert_ids_to_tokens(row["token_ids"].tolist()),
    "old_logprob": row["old_logps"],
    "current_logprob": row["current_logps"],
    "ratio": row["ratio"],
})

The old and current policy use the same weights here, so their ratios should be close to 1. The loss is an inspection value for each batch; no optimizer step is performed.